# Tool-Discovery Agent + Biomni on Colab

In [ ]:
!pip install -q requests beautifulsoup4 pyyaml pandas tabulate python-dateutil langchain-openai biomni

In [ ]:
import os, subprocess, time, sys

REPO_DIR = "/content/scientific_training2_cxy"
if not os.path.exists(REPO_DIR):
    !cd /content && git clone https://github.com/caixiaoyao2025/scientific_training2_cxy.git

os.chdir(REPO_DIR)
os.makedirs("data", exist_ok=True)
print(f"Working in: {os.getcwd()}")

## Start MCP Server & Verify

In [ ]:
env = os.environ.copy()
env["MCP_DATA_ROOT"] = os.path.join(os.getcwd(), "data")
env["MCP_APP_ROOT"] = os.getcwd()
env["MCP_TRANSPORT"] = "streamable-http"
env["MCP_HOST"] = "0.0.0.0"
env["MCP_PORT"] = "8765"
env["MCP_PATH"] = "/mcp"

LOG = os.path.join(os.getcwd(), "server.log")
log_f = open(LOG, "w")
server_proc = subprocess.Popen(
    [sys.executable, "-u", "server.py"],
    env=env, stdout=log_f, stderr=log_f
)
time.sleep(5)

if server_proc.poll() is not None:
    print(f"Server CRASHED (exit code {server_proc.poll()})")
    log_f.close()
    with open(LOG) as f:
        print(f.read()[-2000:])
else:
    print(f"Server running PID={server_proc.pid}")
    import requests
    try:
        r = requests.post("http://127.0.0.1:8765/mcp",
            json={"jsonrpc":"2.0","id":0,"method":"initialize",
                  "params":{"protocolVersion":"2024-11-05","capabilities":{},
                            "clientInfo":{"name":"test","version":"1.0"}}},
            timeout=10)
        print(f"HTTP init: {r.status_code}")
        tools = requests.post("http://127.0.0.1:8765/mcp",
            json={"jsonrpc":"2.0","id":1,"method":"tools/list","params":{}},
            timeout=10).json()
        n = len(tools.get("result",{}).get("tools",[]))
        print(f"Tools loaded: {n}")
        for t in tools.get("result",{}).get("tools",[]):
            print(f"  - {t['name']}")
    except Exception as e:
        print(f"Connection failed: {e}")
        log_f.close()
        with open(LOG) as f:
            print(f.read()[-2000:])

## Run Discovery Pipeline

In [ ]:
from run_pipeline import run_full_pipeline
run_full_pipeline(query="bioinformatics software tool github", max_results=8)

## Connect Biomni

In [ ]:
server_proc.terminate()
server_proc.wait()
print("Standalone server stopped.")

In [ ]:
import yaml
config = {
    "mcp_servers": {
        "bio-mcp": {
            "command": [sys.executable, os.path.join(os.getcwd(), "server.py")],
        }
    }
}
config_path = os.path.join(os.getcwd(), "mcp_config_colab.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f)
print(f"Config: {config_path}")

In [ ]:
from biomni.agent import A1

agent = A1(
    path=os.path.join(os.getcwd(), "data"),
    llm="Qwen/Qwen2.5-14B-Instruct",
    source="Custom",
    base_url="https://api.siliconflow.cn/v1",
    api_key="sk-lufftravmhzgdpudfvbvzwrlfyctebizytmtlbynyiohtkij",
    expected_data_lake_files=[],
)
agent.add_mcp(config_path=config_path)

result = agent.go("List all available MCP tools using list_registered_tools")
print(result)